# para

> Paragraph rules: restatement, elegant variation, forced symmetry, and length

In [ ]:
#| default_exp para

In [ ]:
#| hide
from nbdev.showdoc import *

Paragraph and document rules cover heading repetition, elegant variation, forced symmetry, and paragraph length. The notebook also examines why word-vector similarity cannot reliably distinguish restatement from related content.

In [ ]:
#| export
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *
from slopometer.syntax import *

In [ ]:
from fastcore.test import *

## Word vectors

The vectors in `en_core_web_md` represent each word with 300 numbers learned from word co-occurrences. They come from [GloVe](https://nlp.stanford.edu/projects/glove/) (Pennington, Socher, and Manning, 2014), trained on Common Crawl. Chapter 6 of [Jurafsky and Martin](https://web.stanford.edu/~jurafsky/slp3/) explains the theory.

Words used in similar contexts tend to have similar vectors. Cosine similarity compares their directions: 1.0 means identical directions, and values near 0 indicate little similarity. The comparison is symmetric. It measures relatedness, which does not necessarily mean two words are interchangeable.

spaCy averages token vectors to represent spans and sentences. The experiments below test whether these averages can identify repeated claims. `is_oov` identifies tokens without vectors.

In [ ]:
nlp = get_nlp()
v = nlp.vocab
[(a, b, round(v[a].similarity(v[b]), 3)) for a,b in [('kernel','interpreter'), ('folder','directory'), ('kernel','pineapple')]] + \
    [(w, 'oov', v[w].is_oov) for w in ('heartbeat', 'slopometer', 'exhash')]

[('kernel', 'interpreter', 0.383),
 ('folder', 'directory', 0.323),
 ('kernel', 'pineapple', 0.131),
 ('heartbeat', 'oov', False),
 ('slopometer', 'oov', True),
 ('exhash', 'oov', True)]

"folder" and "directory" score 0.32, while "kernel" and "interpreter" score 0.38. GloVe learned from the general web, where a kernel can be corn and an interpreter a person. A high similarity threshold would miss technical synonyms.

An out-of-vocabulary token need not be jargon. The vocabulary omits project names such as `slopometer` and `exhash`. An invented phrase such as "liveness authority" uses ordinary words. Vocabulary membership cannot establish whether readers understand a term. Slopometer does not score it.

## Restatement

A lead sentence can repeat the rest of its paragraph without adding information. Tell 9 asks writers to remove such sentences.

Compare a paragraph with a redundant lead against one where each sentence adds a fact. If vector similarity can detect restatement, the first paragraph's lead should score higher against the remaining sentences:

In [ ]:
restating = nlp("Failures are visible now too. A startup failure used to print to the server log and leave a half-booted dialog that looked ready. It now raises, and the user sees an error toast and a red status dot.")
progressing = nlp("The ready-wait runs once per kernel, in start. watch polls the process and the heartbeat. Three missed heartbeats mark the kernel unresponsive in its model.")

def _lead_sim(doc):
    sents = list(doc.sents)
    return round(sents[0].similarity(doc[sents[1].start:]), 3)

_lead_sim(restating), _lead_sim(progressing)

(0.857, 0.849)

The scores differ by less than 0.01. Both paragraphs discuss a single topic, which makes their sentence vectors similar. This measurement does not distinguish repeated claims from new facts about the same topic. Lead-sentence restatement remains in the unscoreable registry.

`heading_echo` detects a narrower problem by comparing word lemmas. For example, "Configuration is stored..." repeats the heading "Configuration". This check needs no vectors.

Document-level rules accept all blocks and their parsed `Doc` objects, aligned by position. They return findings with offsets into the whole document.

In [ ]:
#| export
def _content(toks): return {t.lemma_.lower() for t in toks if t.is_alpha and not t.is_stop}

@rule('heading_echo', tell=9, weight=SMELL, level='doc')
def find_heading_echo(blocks, docs):
    "A section's first sentence repeating its heading's words"
    res = []
    for b,nxt,d in zip(blocks, blocks[1:], docs[1:]):
        if b.kind != 'heading' or nxt.kind != 'prose' or d is None: continue
        hw = _content(get_nlp()(scrub(b.txt.lstrip('# '))))
        if not hw: continue
        s1 = list(d.sents)[0]
        lead = first((t.lemma_.lower() for t in s1 if t.is_alpha and not t.is_stop), None)
        if lead in hw and hw <= _content(s1): res.append(Finding('heading_echo', 9, nxt.start, nxt.start+len(s1.text), s1.text, SMELL))
    return res

In [ ]:
#| export
def parse_blocks(blocks):
    "The spaCy `Doc` for each prose block, aligned with `blocks`, None elsewhere"
    prose = [b for b in blocks if b.kind=='prose']
    parsed = iter(get_nlp().pipe([scrub(b.txt) for b in prose]))
    return [next(parsed) if b.kind=='prose' else None for b in blocks]

In [ ]:
echo_doc = "## Configuration\n\nConfiguration is stored in a single file.\n\n## Sockets\n\nEach worker owns one socket.\n"
bs = segment(echo_doc)
res = find_heading_echo(bs, parse_blocks(bs))
test_eq(len(res), 1)
res


[[3] heading_echo (tell 9, restatement): 'Configuration is stored in a single file.']

## Elegant variation

Fowler's "elegant variation" means changing words to avoid repetition. Calling "the heartbeat" "beats" in the next sentence leaves readers to decide whether the terms differ. STE recommends one name per concept. Use the code symbol when one exists: `restart`, not "the respawn operation" or "recycling". Tell 5 applies this guidance to statuses and values.

Vector similarity is a possible detector, but related words need not be interchangeable. Compare synonyms with words for different parts of the same system:

In [ ]:
[(a, b, round(v[a].similarity(v[b]), 3)) for a,b in [('client','server'), ('read','write'), ('heartbeat','beat')]]

[('client', 'server', 0.272),
 ('read', 'write', 0.383),
 ('heartbeat', 'beat', 0.187)]

"client" and "server" score higher than "heartbeat" and "beat". A vector threshold would flag the former while missing the latter.

`find_variation` instead checks curated word pairs and shared stems, such as "beats" inside "heartbeats". It flags the second name. It does not require both nouns to have the same syntactic role. The `write_docs` example uses one as an object and the other as a subject.

The rule ignores tokens made of one repeated character. Segmentation replaces inline code with runs of `x`, and GloVe includes some of these runs in its vocabulary.

In [ ]:
#| export
_variant_pairs = [{'folder', 'directory'}, {'method', 'function'}, {'parameter', 'argument'}, {'error', 'exception'}, {'docstring', 'documentation'}]

@rule('variation', tell=4, weight=SMELL, level='para')
def find_variation(doc):
    "Two names for one concept within a paragraph, by curated pair or shared stem"
    seen,res,done = {},[],set()
    for t in doc:
        if t.pos_ not in ('NOUN', 'PROPN') or not t.is_alpha or len(set(t.lower_)) == 1: continue
        lem = t.lemma_.lower()
        for prev in seen:
            if prev == lem: continue
            pair = any(prev in s and lem in s for s in _variant_pairs)
            stem = (len(min(lem, prev, key=len)) >= 4 and (lem in prev or prev in lem)
                and not doc.vocab[lem].is_oov and not doc.vocab[prev].is_oov)
            if (pair or stem) and frozenset((prev, lem)) not in done:
                done.add(frozenset((prev, lem)))
                res.append(Finding('variation', 4, t.idx, t.idx+len(t.text), f'{seen[prev].text} ... {t.text}', SMELL))
                break
        seen[lem] = t
    return res

In [ ]:
test_eq(find_variation(nlp('The heartbeat channel carries heartbeats only.')), [])
find_variation(nlp('Three missed heartbeats mark it unresponsive, and the next beat clears the mark. Put each parameter on its own line, and document every argument.'))

[[3] variation (tell 4, elegant variation): 'heartbeats ... beat',
 [3] variation (tell 4, elegant variation): 'parameter ... argument']

## Forced symmetry

Tell 20 covers symmetry imposed on the material, such as "fresh ports, fresh channels, and a fresh interpreter" or matching counts of pros and cons. Real material is lumpy.

The meter detects two patterns:

- `triad` finds three or more coordinated nouns with the same adjective lemma in a paragraph.
- `bullet_mold` finds three or more consecutive list items starting with the same word. It operates on document blocks because each list item is a separate block.

In [ ]:
#| export
@rule('triad', tell=20, weight=SMELL, level='para')
def find_triad(doc):
    "Three or more coordinated nouns sharing one modifier lemma"
    res = []
    for t in doc:
        if t.pos_ != 'NOUN': continue
        group = [t] + [c for c in t.conjuncts if c.pos_ == 'NOUN']
        if len(group) < 3 or any(g.i < t.i for g in group): continue
        mods = [{m.lemma_ for m in g.children if m.dep_ == 'amod'} for g in group]
        shared = set.intersection(*mods) if all(mods) else set()
        if shared:
            span = doc.text[group[0].idx:group[-1].idx+len(group[-1].text)]
            res.append(Finding('triad', 20, group[0].idx, group[-1].idx+len(group[-1].text), span, SMELL))
    return res

@rule('bullet_mold', tell=20, weight=SMELL, level='doc')
def find_bullet_mold(blocks, docs):
    "Three or more consecutive list items opening with the same word"
    res,run = [],[]
    def word(b): return re.sub(r'^\s*(?:[-*+]|\d+[.)]) +', '', b.txt).split(' ')[0].lower() if b.txt else ''
    for b in list(blocks) + [None]:
        if b is not None and b.kind == 'item':
            run.append(b)
            continue
        if len(run) >= 3 and len({word(x) for x in run}) == 1:
            res.append(Finding('bullet_mold', 20, run[0].start, run[-1].start+len(run[-1].txt), '\n'.join(x.txt for x in run), SMELL))
        run = []
    return res

In [ ]:
tri = find_triad(nlp('The kernel gains fresh ports, fresh channels, and a fresh interpreter.'))
bs2 = segment('- fast startup\n- fast shutdown\n- fast restarts\n')
tri + find_bullet_mold(bs2, parse_blocks(bs2))


[[3] triad (tell 20, forced symmetry): 'ports, fresh channels, and a fresh interpreter',
 [3] bullet_mold (tell 20, forced symmetry): '- fast startup\n- fast shutdown\n- fast restarts']

## The paragraph cap

ASD-STE100 limits paragraphs to six sentences. A sentence-count rule would penalize splitting a long sentence, contrary to the meter's sentence-length guidance. Slopometer instead flags paragraphs over 150 words, with an additional penalty for each extra 25 words.

An earlier version flagged our calibration example in `write_docs`. It had nine sentences covering both kernel status rules and restart behavior. We split it into two paragraphs instead of exempting it from the rule.


In [ ]:
#| export
@rule('para_cap', tell=None, weight=PRESSURE, level='para')
def find_para_cap(doc):
    "Paragraphs past 150 words, escalating per extra 25"
    n = sum(1 for t in doc if t.is_alpha)
    if n <= 150: return []
    return [Finding('para_cap', None, 0, len(doc.text), f'{n}-word paragraph', PRESSURE*((n-150)//25 + 1))]


In [ ]:
s25 = 'The gateway holds one socket open for every kernel it manages and closes each one when that kernel finally exits. '
test_eq(find_para_cap(nlp(s25 * 6)), [])
test_eq(find_para_cap(nlp('The server does a thing. ' * 9)), [])
find_para_cap(nlp(s25 * 8))


[[1] para_cap: '160-word paragraph']

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()